# 专题分析

In [1]:
import pandas as pd
import numpy as np
import networkx as nx
import scipy
import matplotlib.pyplot as plt
plt.rcParams['font.sans-serif'] = ['Heiti TC']
plt.rcParams['axes.unicode_minus'] = False
from networkx.algorithms import bipartite
import json

In [2]:
import sys
sys.path.append("..")

# 数据处理

In [3]:
df_raw = pd.read_csv("../data/movie_data_rated.csv")
df_raw['k_cast_id'] = df_raw['k_cast_id'].apply(lambda x: str(x))
df_raw

,cast_id,cast_name,gender,movie_id,k_role,is_main_cast,index,id,k_title,k_type,...,rating_num,rating_people,k_rating_people,rating_index,k_movie_id,k_cast_id,uid,portray,movie_id_m,cast_role_agg
0,1017182,陈燕燕,女,10771202,演员,是,1,10771202,深闺疑云,电影,...,7.8,92,92.0,27.0,21542453,2034413,54,赵兰,m21542453,演员
1,1043845,刘志荣,男,10581318,演员,是,11,10581318,四大家族之龙虎兄弟,电影,...,7.8,2290,2290.0,134.0,21162685,2087739,75,NaN,m21162685,导演/演员
2,1050373,茅瑛,女,10573512,演员,是,1,10573512,鬼娘子,电影,...,5.8,235,235.0,37.0,21147073,2100795,79,NaN,m21147073,演员
3,1124360,利智,女,10581318,演员,是,3,10581318,四大家族之龙虎兄弟,电影,...,7.8,2290,2290.0,134.0,21162685,2248769,94,NaN,m21162685,演员
4,1314471,王挺,男,10531190,演员,是,1,10531190,惊天动地,电视剧,...,8.5,326,326.0,53.0,21062429,2628991,147,NaN,m21062429,导演/演员
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
258254,1354509,林泳淘,女,26984991,演员,否,999,26984991,婚姻合伙人,电视剧,...,5.8,1259,1259.0,85.0,53970031,2709067,608462,邓秀玉,m53970031,演员
258255,27496534,刘芊蒂,女,2337597,演员,是,2,2337597,操行零分,电影,...,7.0,211,211.0,38.0,4675243,54993117,608466,NaN,m4675243,演员
258256,27565771,董亚春,女,2270516,导演,是,2,2270516,八月一日,电影,...,6.4,1123,1123.0,85.0,4541081,55131591,608467,NaN,m4541081,导演
258257,35415094,张可,女,7053738,演员,否,999,7053738,樱桃,电视剧,...,5.7,2003,2003.0,107.0,14107525,70830237,608472,NaN,m14107525,演员


# 作品筛选

In [40]:
# 邢冬冬，杨羽
k_cast_id = ["55130387", "55112891"]
# 杨志刚
# k_cast_id = ["54990647"]

In [41]:
# 包含k_cast_id的所有movie_id_m，含这些movie_id_m的所有记录
df_cast_movies = df_raw[df_raw['k_cast_id'].isin(k_cast_id)][['movie_id_m']].drop_duplicates()
df_cast_related = df_raw.merge(df_cast_movies,
                              left_on='movie_id_m',
                              right_on='movie_id_m',
                              how='inner')
df_cast_related

,cast_id,cast_name,gender,movie_id,k_role,is_main_cast,index,id,k_title,k_type,...,rating_num,rating_people,k_rating_people,rating_index,k_movie_id,k_cast_id,uid,portray,movie_id_m,cast_role_agg
0,30301886,王冬,男,20431716,演员,是,4,20431716,男人的刀,电视剧,...,7.5,237,237.0,42.0,40863481,60603821,586,NaN,m40863481,演员
1,27550212,汪小壹,女,26889177,演员,是,10,26889177,异物志,电视剧,...,8.5,82091,82091.0,835.0,53778403,55100473,3914,NaN,m53778403,导演/演员
2,27488967,王秀月,女,5250330,演员,否,999,5250330,快乐的小2B,电视剧,...,7.6,878,878.0,82.0,10500709,54977983,6319,NaN,m10500709,演员
3,27550913,包志强,男,5250330,演员,否,999,5250330,快乐的小2B,电视剧,...,7.6,878,878.0,82.0,10500709,55101875,6717,NaN,m10500709,演员
4,30161546,车志刚,男,5250330,演员,否,999,5250330,快乐的小2B,电视剧,...,7.6,878,878.0,82.0,10500709,60323141,8443,NaN,m10500709,导演/演员
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
406,34930412,李文帅,男,36382628,演员,否,999,36382628,海市蜃楼,电视剧,...,8.1,24200,24200.0,443.0,72765305,69860873,600733,耿师爷,m72765305,演员
407,27500467,邵庄,男,21350566,演员,是,3,21350566,麻辣隔壁·贰,电视剧,...,8.6,3874,3874.0,183.0,42701181,55000983,601559,NaN,m42701181,演员
408,27503136,韩栋,男,36424178,演员,是,1,36424178,真相背后,电视剧,...,6.0,3313,3313.0,141.0,72848405,55006321,601813,NaN,m72848405,演员
409,35201466,付余,女,33438451,演员,否,999,33438451,城市的边缘,电视剧,...,8.3,61858,61858.0,717.0,66876951,70402981,602164,NaN,m66876951,演员


In [42]:
df_cast_related['k_cast_id'].nunique()

142

In [52]:
df_cast_related[df_cast_related['cast_name'] == '纪登超']

,cast_id,cast_name,gender,movie_id,k_role,is_main_cast,index,id,k_title,k_type,...,rating_num,rating_people,k_rating_people,rating_index,k_movie_id,k_cast_id,uid,portray,movie_id_m,cast_role_agg
81,1443206,纪登超,男,4685427,演员,否,999,4685427,大学生同居的事儿前传之歪打歪着,电影,...,7.6,283,283.0,46.0,9370903,2886461,111696,小飞,m9370903,演员
126,1443206,纪登超,男,3794323,演员,是,7,3794323,大学生同居的事儿 第一季,电视剧,...,7.8,1501,1501.0,108.0,7588695,2886461,177474,小飞,m7588695,演员
198,35184121,纪登超,男,4294109,演员,是,8,4294109,大学生同居的事儿 第三季,电视剧,...,7.7,499,499.0,62.0,8588267,2886461,277770,小飞,m8588267,演员
284,1443206,纪登超,男,4122184,演员,是,7,4122184,大学生同居的事儿 第二季,电视剧,...,7.5,570,570.0,65.0,8244417,2886461,395321,小飞,m8244417,演员
285,1443206,纪登超,男,35166815,演员,否,999,35166815,麻辣宿舍,电视剧,...,6.9,2618,2618.0,134.0,70333679,2886461,395322,NaN,m70333679,演员
374,35184121,纪登超,男,5250330,演员,否,999,5250330,快乐的小2B,电视剧,...,7.6,878,878.0,82.0,10500709,2886461,533261,NaN,m10500709,演员


In [44]:
df_casts = df_cast_related[['k_cast_id', 'cast_name']].drop_duplicates()
# 每个cast_name的k_cast_id计数
df_casts_count = df_casts['cast_name'].value_counts().reset_index()
df_casts_count

,cast_name,count
0,纪登超,2
1,毕晨曦,2
2,刘建民,2
3,张晶晶,1
4,韩旭,1
...,...,...
134,杨羽,1
135,刘占奎,1
136,王珺,1
137,冯俊杰,1


In [45]:
# 将出现多个k_cast_id的cast_name，统一为第一个k_cast_id
cast_name_to_id = {}
for _, row in df_casts.iterrows():
    name = row['cast_name']
    k_id = row['k_cast_id']
    if name not in cast_name_to_id:
        cast_name_to_id[name] = k_id
df_cast_related['k_cast_id'] = df_cast_related['cast_name'].apply(lambda x: cast_name_to_id[x])
df_cast_related['k_cast_id'].nunique()

139

In [46]:
# 删除特定作品
movie_to_remove = ["山海经之山河图"] 
df_cast_related = df_cast_related[~df_cast_related['k_title'].isin(movie_to_remove)]

In [47]:
df_cast_related['movie_id_m'].nunique()

30

In [48]:
df_cast_related['k_title'].unique()

array(['男人的刀', '异物志', '快乐的小2B', '冤家宜解不宜结', '金牌保镖', '毛骗 终结篇', '一起同过窗 第三季',
       '城市的边缘', '输不起', '毛骗 第二季', '真相背后', '大学生同居的事儿前传之歪打歪着', '麻辣隔壁',
       '麻辣宿舍', '热血篮球', '毛骗 第一季', '大学生同居的事儿 第三季', '那些特立独行的猪', '海市蜃楼',
       '杀不死', '大学生同居的事儿 第二季', '麻辣隔壁·叁', '一屋高才生', '男人不醉', '麻辣兄弟之疯狂一夜',
       '麻辣隔壁·肆', '大学生同居的事儿 第一季', '非常保镖', '第三个人', '麻辣隔壁·贰'], dtype=object)

In [49]:
df_cast_related['movie_id'].unique()

array([20431716, 26889177,  5250330, 35150248, 35744460, 26603847,
       34989151, 33438451, 30168034,  6894818, 36424178,  4685427,
       11527516, 35166815, 26438944,  4888230,  4294109, 11610647,
       36382628, 26772013,  4122184, 25892766, 26617559, 26296022,
       35204870, 26617283,  3794323, 35176575, 25830922, 21350566])

In [50]:
df_cast_related[df_cast_related['k_title'] == '第三个人']

,cast_id,cast_name,gender,movie_id,k_role,is_main_cast,index,id,k_title,k_type,...,rating_num,rating_people,k_rating_people,rating_index,k_movie_id,k_cast_id,uid,portray,movie_id_m,cast_role_agg
151,27565169,邢冬冬,男,25830922,演员,否,999,25830922,第三个人,电影,...,7.3,105,105.0,28.0,51661893,55130387,208584,NaN,m51661893,导演/演员
250,27502635,安宁,女,25830922,演员,是,2,25830922,第三个人,电影,...,7.3,105,105.0,28.0,51661893,55005319,349839,NaN,m51661893,演员
329,35101242,姚梦,女,25830922,演员,否,999,25830922,第三个人,电影,...,7.3,105,105.0,28.0,51661893,70202533,462861,欣欣,m51661893,演员
358,27493552,蔡明,男,25830922,导演,是,1,25830922,第三个人,电影,...,7.3,105,105.0,28.0,51661893,54987153,494528,NaN,m51661893,导演/演员
363,27556421,杨羽,男,25830922,演员,否,999,25830922,第三个人,电影,...,7.3,105,105.0,28.0,51661893,55112891,514662,NaN,m51661893,导演/演员


In [51]:
# 保存为csv
df_cast_related.to_csv(f"youyin_cast_works.csv", index=False)

In [ ]:
movie_list = df_cast_related[[
    'k_title', 'movie_id_m', 'k_movie_year', 'k_type', 'rating_num',
    'k_rating_people'
]].drop_duplicates().sort_values(by='k_movie_year').reset_index(drop=True)

In [ ]:
movie_list.to_csv("youyin_cast_movie_list.csv", index=False)

In [ ]:
import plotly.express as px

In [ ]:
fig = px.scatter(movie_list,
                 x="k_rating_people",
                 y="rating_num",
                #  size="rating_num",
                 color="k_type",
                 hover_name="k_title",
                 log_x=True,
                 size_max=60)
fig.show()